# CPU Baseline — Non-Maximum Suppression (NMS)
CSC14116 — Applied Parallel Programming, Topic A4.

Greedy NMS implemented in pure NumPy. This is the serial O(n^2) reference that the GPU kernels (V1/V2/V3) in this project are benchmarked against, and the bottleneck `cProfile` should point at below.

Mirrors `cpu_baseline.py` — same `load_data` / `run_cpu` / `verify` / `benchmark` functions, runnable directly on Colab/Kaggle without a GPU.

In [ ]:
import time

import numpy as np

## load_data
Synthetic (boxes, scores) generator for benchmarking, plus an optional loader for real YOLOv5s detections.

In [ ]:
def load_data(n, seed=0):
    """Generate synthetic (boxes, scores).

    boxes: (n, 4) array of [x1, y1, x2, y2]
    scores: (n,) array of confidence scores in [0, 1)
    """
    rng = np.random.default_rng(seed)
    x1 = rng.uniform(0, 900, size=n)
    y1 = rng.uniform(0, 900, size=n)
    w = rng.uniform(10, 100, size=n)
    h = rng.uniform(10, 100, size=n)
    boxes = np.stack([x1, y1, x1 + w, y1 + h], axis=1).astype(np.float32)
    scores = rng.uniform(0, 1, size=n).astype(np.float32)
    return boxes, scores

In [ ]:
def load_real_boxes(image_paths=None, conf_threshold=0.25):
    """Run pretrained YOLOv5s on a handful of images to get real (boxes, scores)."""
    import torch

    model = torch.hub.load("ultralytics/yolov5", "yolov5s", pretrained=True)
    model.conf = conf_threshold
    if not image_paths:
        image_paths = ["https://ultralytics.com/images/zidane.jpg"]

    results = model(image_paths)
    boxes_list = [pred[:, :4].cpu().numpy() for pred in results.xyxy]
    scores_list = [pred[:, 4].cpu().numpy() for pred in results.xyxy]
    boxes = np.concatenate(boxes_list, axis=0).astype(np.float32)
    scores = np.concatenate(scores_list, axis=0).astype(np.float32)
    return boxes, scores

## run_cpu
Greedy NMS: sort by score, then sequentially keep/suppress. Kept intentionally serial (not vectorized across the suppression loop) since that sequential dependency is exactly what GPU V3 (Matrix NMS) is meant to remove — see *The Challenge* in `PROPOSAL_DRAFT.md`.

In [ ]:
def iou_one_to_many(box, boxes):
    """IoU between a single box (4,) and an array of boxes (M, 4)."""
    xx1 = np.maximum(box[0], boxes[:, 0])
    yy1 = np.maximum(box[1], boxes[:, 1])
    xx2 = np.minimum(box[2], boxes[:, 2])
    yy2 = np.minimum(box[3], boxes[:, 3])

    inter_w = np.maximum(0.0, xx2 - xx1)
    inter_h = np.maximum(0.0, yy2 - yy1)
    inter = inter_w * inter_h

    area_box = (box[2] - box[0]) * (box[3] - box[1])
    area_boxes = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])

    union = area_box + area_boxes - inter
    return inter / np.maximum(union, 1e-9)

In [ ]:
def run_cpu(boxes, scores, iou_threshold=0.5):
    order = np.argsort(-scores, kind="stable")
    suppressed = np.zeros(len(boxes), dtype=bool)
    keep = []

    for i in range(len(order)):
        idx = order[i]
        if suppressed[idx]:
            continue
        keep.append(idx)

        remaining = order[i + 1:]
        remaining = remaining[~suppressed[remaining]]
        if len(remaining) == 0:
            continue

        ious = iou_one_to_many(boxes[idx], boxes[remaining])
        suppressed[remaining[ious > iou_threshold]] = True

    return np.array(keep, dtype=np.int64)

## verify
Compare our NMS output against `torchvision.ops.nms` (ground truth). Requires `torch`/`torchvision` — available by default on Colab.

In [ ]:
def verify(boxes, scores, iou_threshold, keep):
    try:
        import torch
        from torchvision.ops import nms as torch_nms
    except ImportError:
        print("torchvision not installed -- skipping verification against reference NMS")
        return None

    ref_keep = torch_nms(
        torch.from_numpy(boxes), torch.from_numpy(scores), iou_threshold
    ).numpy()

    ours, theirs = set(keep.tolist()), set(ref_keep.tolist())
    matches = ours == theirs
    print(f"Reference (torchvision) kept {len(theirs)} boxes, ours kept {len(ours)}")
    print(f"Exact match: {matches}")
    if not matches:
        print(f"  only ours:   {sorted(ours - theirs)}")
        print(f"  only theirs: {sorted(theirs - ours)}")
    return matches

## benchmark
Time `run_cpu` across increasing N to find where it becomes a bottleneck.

In [ ]:
def benchmark(ns=(100, 1000, 10000), iou_threshold=0.5, seed=0):
    print(f"{'N':>8} | {'time (s)':>10}")
    print("-" * 21)
    results = {}
    for n in ns:
        boxes, scores = load_data(n, seed=seed)
        start = time.perf_counter()
        run_cpu(boxes, scores, iou_threshold)
        elapsed = time.perf_counter() - start
        results[n] = elapsed
        print(f"{n:>8} | {elapsed:>10.4f}")
    return results

## Demo run + benchmark sweep

In [ ]:
boxes, scores = load_data(1000, seed=0)
keep = run_cpu(boxes, scores, iou_threshold=0.5)
print(f"NMS kept {len(keep)}/{len(boxes)} boxes")
verify(boxes, scores, 0.5, keep)

In [ ]:
_ = benchmark()

## Profiling (for Background/Challenge section of the proposal)

In [ ]:
import cProfile
import pstats

boxes, scores = load_data(10000, seed=0)
profiler = cProfile.Profile()
profiler.enable()
run_cpu(boxes, scores, 0.5)
profiler.disable()
pstats.Stats(profiler).sort_stats("cumulative").print_stats(10)